In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
import os
os.environ["OPENAI_BASE_URL"]


'https://generativelanguage.googleapis.com/v1beta/openai/'

In [4]:
os.environ["GOOGLE_API_KEY"] = os.environ["OPENAI_API_KEY"]

In [5]:
from langchain.messages import HumanMessage

class Conversation:
    def __init__(self, agent, thread_id) -> None:
        self.agent = agent
        self.thread_id = thread_id
        pass
    async def message(self, message):
        question = HumanMessage(content=message)
        config = {"configurable": {"thread_id": self.thread_id}}

        response = await self.agent.ainvoke(
            {"messages": [question]},
            config,  
        )
        return response



In [6]:
import os
from langchain.chat_models import init_chat_model


AGENT_LLM_NAME = "gemini-2.5-flash"
model = init_chat_model(f"google_genai:{AGENT_LLM_NAME}")




None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [7]:
from src.eda_rajiv.app_sqlite import CODE_INTERPRETER_INSTRUCTIONS, WOMAN_SPORTS_PURCHASE_QUERY

In [8]:
from pprint import pprint
pprint(CODE_INTERPRETER_INSTRUCTIONS)

('The `web_search` tool takes english like input and return back english like '
 "output. Use that to improve your dialect's queries or understand more about "
 'the financial domain.\n'
 'The `code_interpreter` tool executes SQL queries and returns back a JSON '
 'string with stdout and stderr Your output is an SQL query without any '
 'markdown or surrounding quotes for a SQLite database.\n'
 'You can access the local filesystem using this tool. The data is in '
 '`/data/fintran.db`.  Any query from the user should use this file.\n'
 '\n'
 'Recommended packages: Pandas, Numpy, SymPy, Scikit-learn.\n'
 'You can also run Jupyter-style shell commands (e.g., `!pip freeze`)\n'
 "but you won't be able to install packages.\n"
 '\n'
 '\n'
 'Few Shot Examples:\n'
 'Question: "What were the average transactions for spending on sports?"\n'
 'Answer: SELECT AVG(t.transaction_amount) FROM transactions t JOIN mcc_codes '
 "m ON t.mcc = m.mcc_code WHERE m.description LIKE '% sport %' OR "
 "m.descr

In [9]:
# from langchain.messages import SystemMessage, HumanMessage
# messages = [
#     SystemMessage(
#         content=CODE_INTERPRETER_INSTRUCTIONS
#     ),
#     HumanMessage(content=WOMAN_SPORTS_PURCHASE_QUERY),
# ]

# response = model.invoke(messages)
# response 


# Tools


## Web Search

In [10]:
from src.utils import pretty_print
from src.utils.tools.gemini_grounding import GeminiGroundingWithGoogleSearch


tool_cls = GeminiGroundingWithGoogleSearch()
response = await tool_cls.get_web_search_grounded_response(
    "How does the annual growth in the 50th-percentile income "
    "in the US compare with that in Canada?"
)

pretty_print(response.text_with_citations)


"The annual growth in the 50th-percentile income, often represented by the median household income, has shown varying trends in the US and Canada in recent years.\n\nIn the United States, the real median household income increased by 2.0% from $80,000 in 2023 to $81,600 in 2024 (in 2024 inflation-adjusted dollars).[1](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHDHUCzvOhIAENZKXtwrwGhWY_eE0ZbBYLx2HN-mMAvCONkN9olbLoxqVGCfWBq16n4CiNWuiPf7PG4o8Q9dND8V95t-9qULzwVCqNESAtl4E16bNW2KTY1I0BTkn--llAZ37yKfKE8pYjp0ZDc29SDCvkl_EKmkc-fM1ZkHVhIlqqB979jWbSJQFSRHjT3wO2wrg==) However, another report from the U.S. Census Bureau indicates that the real median household income in 2024, at $83,730, was not statistically different from the 2023 estimate of $82,690 (both in 2024 inflation-adjusted dollars), suggesting a very small or no statistically significant real growth during this period.[2](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEDdUuMWhdtGlMLqsxiPeZ

'"The annual growth in the 50th-percentile income, often represented by the median household income, has shown varying trends in the US and Canada in recent years.\\n\\nIn the United States, the real median household income increased by 2.0% from $80,000 in 2023 to $81,600 in 2024 (in 2024 inflation-adjusted dollars).[1](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHDHUCzvOhIAENZKXtwrwGhWY_eE0ZbBYLx2HN-mMAvCONkN9olbLoxqVGCfWBq16n4CiNWuiPf7PG4o8Q9dND8V95t-9qULzwVCqNESAtl4E16bNW2KTY1I0BTkn--llAZ37yKfKE8pYjp0ZDc29SDCvkl_EKmkc-fM1ZkHVhIlqqB979jWbSJQFSRHjT3wO2wrg==) However, another report from the U.S. Census Bureau indicates that the real median household income in 2024, at $83,730, was not statistically different from the 2023 estimate of $82,690 (both in 2024 inflation-adjusted dollars), suggesting a very small or no statistically significant real growth during this period.[2](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEDdUuMWhdtGlMLqsxi

## Setup Interpreter


In [11]:
from pathlib import Path
from src import eda_rajiv
from src.eda_rajiv.finance_data_code_interpreter import  make_python_code_interpreter, make_fintran_db_code_interpreter
init_module_path = Path(eda_rajiv.__file__).parent / "sql.py"
python_code_interpreter = await make_python_code_interpreter()
a_code_interpreter = await make_fintran_db_code_interpreter(init_module_path=init_module_path) 



In [12]:
from langchain.tools import tool

@tool
async def code_interpreter(query: str) -> str:
    """Run the SQL Query in a sandbox and return a JSON string of the stdout and stderr"""
    return await a_code_interpreter.run_query(query)

In [13]:
@tool
async def web_search(query: str) -> str:
    "Run a Web Query and get an English Response"
    response = await tool_cls.get_web_search_grounded_response(query)
    return response.text_with_citations



In [14]:
# Example async invocation of the tool
response = await code_interpreter.ainvoke({"query": "select count(*) from users_data"})
response

===== Run Query =======
'select count(*) from users_data'


'{"stdout":["{","  \\"columns\\": [","    \\"count(*)\\"","  ],","  \\"data\\": [","    [","      2000","    ]","  ]","}"],"stderr":[],"error":null}'

In [15]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
# system_prompt="You are a fictional writer's assistant. If asked a non factual question, make things up in one sentence"

agent = create_agent(
    model,
    tools=[code_interpreter, web_search],
    system_prompt=CODE_INTERPRETER_INSTRUCTIONS,
    checkpointer=InMemorySaver()
)


In [16]:
from langchain.messages import HumanMessage

result = []


c = Conversation(agent, "1")

In [18]:
r = await c.message("What are the MCC codes related to sports?")

===== Run Query =======
("SELECT id, description FROM mcc_codes WHERE description LIKE '% sport %' OR "
 "description LIKE 'Sport %' OR description LIKE '% Sport' OR description LIKE "
 "'Sport';")
===== Run Query =======
("SELECT id, description FROM mcc_codes WHERE id IN ('7941', '5941', '7992', "
 "'7997');")
===== Run Query =======
("SELECT id, description FROM mcc_codes WHERE description LIKE '%Sporting "
 "Goods%' OR description LIKE '%Commercial Sports%' OR description LIKE "
 "'%Athletic Fields%' OR description LIKE '%Sport Clubs%' OR description LIKE "
 "'%Sport Promoters%' OR description LIKE '%Golf Courses%' OR description LIKE "
 "'%Country Clubs%' OR description LIKE '%Private Clubs%';")


In [19]:
r

{'messages': [HumanMessage(content='What are the MCC codes related to sports?', additional_kwargs={}, response_metadata={}, id='33ad3e7c-cfa7-4e31-816d-d108af4ff099'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'code_interpreter', 'arguments': '{"query": "SELECT id, description FROM mcc_codes WHERE description LIKE \'% sport %\' OR description LIKE \'Sport %\' OR description LIKE \'% Sport\' OR description LIKE \'Sport\';"}'}, '__gemini_function_call_thought_signatures__': {'509c2705-a76d-4ee0-8537-e7e3dfb0f8bd': 'CpsCAXLI2nxLIGqL6OIE9qRcWylquYGeU/y60jVttcxuXYeaXoCdnFhHE9I8AHCr8RfX565EagdhlC5ZkFJ8MMCvA0Z0X/fnirxZ1pIDhxdCsnS/BVoCG7jp7rN/FIMuxzaS7ARfh7DHUX77EDewfBTcxKlSDtYDEX2+Zit3lgy9oiauVtbLsSCuxMOTHhukWWvRi2/r07D2rQKxIYWRo4aGVRYR2iQHevKlvgopRWaOF03kBoQWOTOxf8AcfBJYypyerTReqQ1ycOdEFhgC35UJVmVvTTf2Hu1KgDZ2Bd39kLGhjHOhD4UZfB31zqkyp9hABjjAGFK2Ou8vFzMjAyxw3J8FuOBqxOluvKCxBe0x/k+LCmqeCbcPurPgkg=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini

In [20]:
r['messages'][-1].content

[{'type': 'text',
  'text': 'The MCC codes related to sports are:\n*   **7801**: Athletic Fields, Commercial Sports\n*   **5941**: Sporting Goods Stores',
  'extras': {'signature': 'CtwBAXLI2nwSIX6bS0AkdqSQ3Z1BlKm77aYHqy9pv0WzqFVZAsw+gqgiWDXW5dcfhjSije2EGOxAHJNRHWfuAYqjs9S5I6i34qSviKcYkw+BdgJKgcyND3BdH4qB6GhuXCCFvlDZBKTXZ96sjir6PKu4/GSJbexS0woRhTsjMCsj+h2A7c0HMenqiXn8k1gJ3dP8Wx1N63fxScvBk5ZQ27tSrF1v+gmJ01PuMT8adW24fxyqwcH+aebVJBORt85NLjDn8KM0CyC0PUIgWBO462ImFchEce1D80UodEnTyw=='}}]

In [21]:
r = await c.message(WOMAN_SPORTS_PURCHASE_QUERY)


===== Run Query =======
('SELECT SUM(t.amount) FROM transactions_data t JOIN users_data u ON '
 't.client_id = u.id JOIN mcc_codes m ON t.mcc = m.id WHERE u.gender = '
 "'Female' AND (m.id = '7801' OR m.id = '5941');")


In [22]:
r

{'messages': [HumanMessage(content='What are the MCC codes related to sports?', additional_kwargs={}, response_metadata={}, id='33ad3e7c-cfa7-4e31-816d-d108af4ff099'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'code_interpreter', 'arguments': '{"query": "SELECT id, description FROM mcc_codes WHERE description LIKE \'% sport %\' OR description LIKE \'Sport %\' OR description LIKE \'% Sport\' OR description LIKE \'Sport\';"}'}, '__gemini_function_call_thought_signatures__': {'509c2705-a76d-4ee0-8537-e7e3dfb0f8bd': 'CpsCAXLI2nxLIGqL6OIE9qRcWylquYGeU/y60jVttcxuXYeaXoCdnFhHE9I8AHCr8RfX565EagdhlC5ZkFJ8MMCvA0Z0X/fnirxZ1pIDhxdCsnS/BVoCG7jp7rN/FIMuxzaS7ARfh7DHUX77EDewfBTcxKlSDtYDEX2+Zit3lgy9oiauVtbLsSCuxMOTHhukWWvRi2/r07D2rQKxIYWRo4aGVRYR2iQHevKlvgopRWaOF03kBoQWOTOxf8AcfBJYypyerTReqQ1ycOdEFhgC35UJVmVvTTf2Hu1KgDZ2Bd39kLGhjHOhD4UZfB31zqkyp9hABjjAGFK2Ou8vFzMjAyxw3J8FuOBqxOluvKCxBe0x/k+LCmqeCbcPurPgkg=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini